# Apigee Template: REST-AI-Completions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-template-repository/blob/main/notebooks/REST-AI-Completions.ipynb)

**Template Name:** `REST-AI-Completions`  
**Description:** Multi-provider AI Chat Completions API Gateway proxy with model routing, token counting, and API key validation.

### Workflow:
1. **Configuration & Authentication**: Enter your Google Cloud Project ID and authenticate.
2. **Setup & Deploy**: Initialize GCP/Apigee resources and deploy `REST-AI-Completions.yaml` using `aft`.
3. **Test AI Routing**: Send chat completion requests to Gemini, OpenAI, and Anthropic targets.

---

## 1. Configuration & Authentication

Specify your Google Cloud Project ID and Apigee environment, then authenticate.

In [ ]:
# @title 1. Configuration & Authentication
import os

# @markdown Enter your Google Cloud Project ID (Apigee Organization):
PROJECT_ID = "your_apigee_org"  # @param {type:"string"}
APIGEE_ENV = "dev"  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["APIGEE_ORG"] = PROJECT_ID
os.environ["APIGEE_ENV"] = APIGEE_ENV

try:
    from google.colab import auth
    auth.authenticate_user()
    print(f"Authenticated with Google Cloud for project: {PROJECT_ID}")
except ImportError:
    print("Running outside Google Colab.")


## 2. Setup Tools, Initialize Resources & Deploy Template

Downloads `REST-AI-Completions.yaml` and `sh/initialize.sh` (if running standalone), installs the `aft` CLI, runs initialization (service account, IAM roles, data collectors, reports), and deploys the template.

In [ ]:
# @title 2. Setup & Deploy
import os

# 1. Install Apigee Feature Templater (aft) CLI if needed
!which aft >/dev/null 2>&1 || curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh

# 2. Download template and initialize script if running standalone
REPO_RAW = "https://raw.githubusercontent.com/gcp-samples/apigee-template-repository/main"
TEMPLATE_FILE = "REST-AI-Completions.yaml" if os.path.exists("REST-AI-Completions.yaml") else "templates/REST-AI-Completions.yaml" if os.path.exists("templates/REST-AI-Completions.yaml") else "REST-AI-Completions.yaml"

if not os.path.exists(TEMPLATE_FILE):
    !curl -fsSL -O {REPO_RAW}/templates/REST-AI-Completions.yaml

if not os.path.exists("sh/initialize.sh"):
    !mkdir -p sh && curl -fsSL -o sh/initialize.sh {REPO_RAW}/sh/initialize.sh

# 3. Run initialization (creates service account, IAM bindings, data collectors, reports)
!bash sh/initialize.sh

# 4. Deploy template with aft
!aft {TEMPLATE_FILE} \
  --organization="$APIGEE_ORG" \
  --environment="$APIGEE_ENV" \
  --service-account="apigee-service@${GOOGLE_CLOUD_PROJECT}.iam.gserviceaccount.com"


## 3. Test AI Completions Routing

Send test chat completion requests to verify multi-provider routing for Google Cloud Vertex AI (Gemini), OpenAI (GPT-4o), and Anthropic (Claude).

In [ ]:
# @title Setup Test Client & APIGEE_HOST
import os
import json
import requests
import subprocess

# Retrieve GCP access token for Authorization header
try:
    token = subprocess.check_output(["gcloud", "auth", "application-default", "print-access-token"], text=True).strip()
except Exception:
    token = subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {token}"
}

# Retrieve Apigee environment group hostname
try:
    resp = requests.get(f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/envgroups", headers=headers, timeout=15).json()
    APIGEE_HOST = resp["environmentGroups"][0]["hostnames"][-1]
except Exception:
    APIGEE_HOST = os.getenv("APIGEE_HOST", f"{PROJECT_ID}-{APIGEE_ENV}.apigee.net")

ENDPOINT_URL = f"https://{APIGEE_HOST}/v1/chat/completions"
print(f"Target Endpoint: {ENDPOINT_URL}")

def send_chat(model: str, prompt: str):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}]
    }
    print(f"\nSending [{model}] request to {ENDPOINT_URL}...")
    try:
        res = requests.post(ENDPOINT_URL, headers=headers, json=payload, timeout=30)
        print(f"Status Code: {res.status_code}")
        print(json.dumps(res.json(), indent=2))
    except Exception as e:
        print("Error:", e)


In [ ]:
# @title Test 1: Google Cloud Vertex AI (Gemini 3.6 Flash)
send_chat("gemini-3.6-flash", "Explain quantum computing in one sentence.")


In [ ]:
# @title Test 2: OpenAI Target (GPT-4o)
send_chat("gpt-4o", "What is the capital of France?")


In [ ]:
# @title Test 3: Anthropic Target (Claude 3.5 Sonnet)
send_chat("claude-3-5-sonnet-20240620", "List 3 key benefits of an enterprise API gateway.")
